In [1]:
import sqlite3
import csv
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
db_path = "./data/merged.db"
table_name = "data"
output_csv = "./data/merged.csv"
output_parquet = "./data/merged.parquet"
output_pickle = "./data/merged.pkl"
chunk_size = 10000

In [3]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

In [4]:
query = f"SELECT * FROM {table_name}"
cursor.execute(query)

In [20]:
total = 0
with open(output_csv, "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file, delimiter='\x1F', quoting=csv.QUOTE_NONE, escapechar='\\')
    
    column_names = [desc[0] for desc in cursor.description]
    writer.writerow(column_names)
    
    i = 0
    t = f = 0
    while True:
        rows = cursor.fetchmany(chunk_size)
        if not rows:
            break
        # filtered = [r for r in rows if 500 <= len(r[2]) >= 2000]
        filtered = []
        for r in rows:
            if 100 <= len(r[2]) <= 150000:
                if r[3] == 0:
                    f +=1
                else:
                    t +=1
                filtered.append(r)
        writer.writerows(filtered)
        total += len(filtered)
        if i % 10 == 0:
            print(i + 1, total)
        i += 1
    print('finish', total, t, f)

KeyboardInterrupt: 

In [6]:
parquet_writer = None
    
# 2) 컬럼명
col_names = [desc[0] for desc in cursor.description]

# 3) chunk_size 단위로 fetch
i = 0
t = f = 0
total = 0
while True:
    rows = cursor.fetchmany(chunk_size)
    if not rows:
        break
    
    filtered = []
    for r in rows:
        if 100 <= len(r[2]) <= 10000:
            if r[3] == 0:
                f +=1
            else:
                t +=1
            filtered.append(r)
    
    # Pandas DataFrame으로 변환
    df = pd.DataFrame(rows, columns=col_names)
    
    # Parquet Table로 변환
    table = pa.Table.from_pandas(df)
    
    if parquet_writer is None:
        # 첫 테이블 -> 새로 파일 생성
        parquet_writer = pq.ParquetWriter(output_parquet, table.schema)
    
    # 현재 청크 데이터를 Parquet 파일에 write
    parquet_writer.write_table(table)
    
    total += len(filtered)
    if i % 10 == 0:
        print(i + 1, total)
    i += 1
    
    # df, table 참조 해제
    del df, table

# 종료 작업
if parquet_writer:
    parquet_writer.close()

cursor.close()
conn.close()
print(f"Parquet 저장 완료: {output_parquet}, 0: {f}, 1: {t}, total: {total}")

1 90
11 15808
21 27425
31 50667
41 71558
51 98438
Parquet 저장 완료: ./data/merged.parquet, 0: 45796, 1: 65229, total: 111025


In [5]:
column_names = [desc[0] for desc in cursor.description]

# 4. 청크들을 저장할 임시 리스트
df_chunks = []

i = 0
t = f = 0
total = 0
while True:
    # chunk_size만큼 데이터를 가져오기
    rows = cursor.fetchmany(chunk_size)
    if not rows:
        break  # 더 이상 가져올 데이터가 없으면 종료
    
    filtered = []
    for r in rows:
        if 1000 <= len(r[2]) <= 30000:
            if r[3] == 0:
                f +=1
            else:
                t +=1
            filtered.append(r)
    
    # 이 청크를 DataFrame으로 변환
    df_chunk = pd.DataFrame(filtered, columns=column_names)
    
    # 리스트에 추가
    df_chunks.append(df_chunk)
    
    total += len(filtered)
    if i % 10 == 0:
        print(i + 1, total)
    i += 1

# 5. 모든 청크를 하나의 DataFrame으로 합치기
#    데이터가 매우 클 경우 메모리 초과 위험이 있으니 주의
if df_chunks:
    df_final = pd.concat(df_chunks, ignore_index=True)
else:
    # 혹시라도 데이터가 하나도 없는 경우
    df_final = pd.DataFrame(columns=column_names)

# 6. 피클 파일로 저장
df_final.to_pickle(output_pickle)

# 7. 자원 정리
cursor.close()
conn.close()
print(f"Pickle 저장 완료: {output_pickle}, 0: {f}, 1: {t}, total: {total}")

1 117
11 15560
21 34168
31 53879
41 72821
51 99178
Pickle 저장 완료: ./data/merged.pkl, 0: 51603, 1: 59164, total: 110767
